In [10]:
import pandas as pd
from scripts.stats import friedman_test, wilcoxon_test, wilcoxon_ranking

dfs = {}
results_dir = "results_final"

dataset = "INA_icd"
n_iter = 5

metrics = ["AUC", "F1", "Prec", "Recall", "MCC", "Acc", "Brier"]
alpha = 0.05

friedman_results = {}
wilcoxon_p_matrices = {}
wilcoxon_rankings = {}

for metric in metrics:
    df = pd.read_csv(
        f"{results_dir}/resdata_{metric}_{dataset}_ITER{n_iter}.csv",
        index_col=0
    )

    if "STATIC" in df.columns:
        df = df.drop(columns=["STATIC"])

    if "STATIC+GRU-D" in df.columns:
        df = df.drop(columns=["STATIC+GRU-D"])

    # Invert Brier so that higher is always better
    if metric == "Brier":
        df = 1 - df

    dfs[metric] = df

    stat, p = friedman_test(df, alpha=alpha, debug=True)

    friedman_results[metric] = {
        "statistic": stat,
        "p_value": p,
        "global_diff": "YES" if p < alpha else "NO"
    }

    if p < alpha:
        p_matrix, reject, p_corr, results = wilcoxon_test(
            df,
            alpha=alpha,
            debug=True
        )

        ranking = wilcoxon_ranking(
            df,
            reject,
            higher_is_better=True
        )

        wilcoxon_p_matrices[metric] = p_matrix
        wilcoxon_rankings[metric] = ranking

friedman_df = pd.DataFrame(friedman_results).T


=== FRIEDMAN TEST ===
Statistic: 46.5943
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 43.8956
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 88.0773
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 94.7858
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 29.6569
p-value: 0.000046
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 104.5720
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 21.4286
p-value: 0.001536
➡️ SIGNIFICANT global differences


In [11]:
friedman_latex = friedman_df.copy()

friedman_latex["p_value"] = friedman_latex["p_value"].apply(
    lambda x: f"{x:.2e}"
)

friedman_latex["global_diff"] = friedman_latex["global_diff"].map({
    "YES": r"\checkmark",
    "NO": r"\ding{55}"
})

friedman_latex = friedman_latex.rename(columns={
    "statistic": "Statistic",
    "p_value": "$p$-value",
    "global_diff": "Difference"
})

friedman_latex = friedman_latex.T
latex_table = friedman_latex.to_latex(
    escape=False,
    caption="Friedman test results for all evaluation metrics.",
    label="tab:friedman",
    column_format="l" + "c"*len(friedman_latex.columns)
)

print(latex_table)

\begin{table}
\caption{Friedman test results for all evaluation metrics.}
\label{tab:friedman}
\begin{tabular}{lccccccc}
\toprule
 & AUC & F1 & Prec & Recall & MCC & Acc & Brier \\
\midrule
Statistic & 46.594286 & 43.895640 & 88.077253 & 94.785767 & 29.656898 & 104.572049 & 21.428571 \\
$p$-value & 2.25e-08 & 7.75e-08 & 7.60e-17 & 3.06e-18 & 4.57e-05 & 2.78e-20 & 1.54e-03 \\
Difference & \checkmark & \checkmark & \checkmark & \checkmark & \checkmark & \checkmark & \checkmark \\
\bottomrule
\end{tabular}
\end{table}



In [12]:
selected_metrics = ["MCC", "F1", "Prec", "Recall", "AUC"]

ranking_table = pd.DataFrame()

for metric in selected_metrics:

    ranking = wilcoxon_rankings[metric].copy()

    ranking["rank"] = ranking["score"].rank(
        method="min",
        ascending=False
    ).astype(int)

    ranking_table[metric] = ranking["rank"]

ranking_table = ranking_table[selected_metrics]

# ordinamento gerarchico:
ranking_table = ranking_table.sort_values(
    by=selected_metrics,
    ascending=True      # rank 1 migliore
)
print(ranking_table.to_latex())

\begin{tabular}{lrrrrr}
\toprule
 & MCC & F1 & Prec & Recall & AUC \\
\midrule
STATIC+Dipole & 1 & 1 & 1 & 5 & 3 \\
STATIC+BiPadLSTM & 1 & 1 & 2 & 5 & 6 \\
STATIC+GRU & 1 & 1 & 4 & 3 & 3 \\
STATIC+Med2Vec & 1 & 1 & 4 & 4 & 2 \\
STATIC+DOME & 1 & 1 & 6 & 1 & 1 \\
STATIC+CEHR-BERT & 6 & 1 & 2 & 5 & 6 \\
STATIC+EVENT-CNT & 7 & 7 & 7 & 2 & 3 \\
\bottomrule
\end{tabular}

